<a href="https://colab.research.google.com/github/hjiwoong/subway/blob/main/%EC%84%9C%EC%9A%B8%EC%8B%9C_%EC%A7%80%ED%95%98%EC%B2%A0_%EA%B5%90%ED%86%B5%EB%9F%89_%EC%98%88%EC%B8%A1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### 환경설정

https://data.seoul.go.kr/dataList/OA-12252/S/1/datasetView.do

In [87]:
#구글 드라이브 마운트
from google.colab import drive
drive.mount('/content/drive')

#드라이브에 저장된 폰트 등록
import matplotlib as mpl
import matplotlib.pyplot as plt #그래프 그리는 모듈
import matplotlib.font_manager as fm #폰트 관리하는 모듈

#드라이브 내 폰트 경로
font_path = "/content/drive/MyDrive/KWU/bigdata/dataPreProcessing/NanumGothic.ttf"

fm.fontManager.addfont(font_path)
mpl.rc('font',family = 'NanumGothic') #matplolib 기본 폰트로 설정
plt.rcParams['font.family'] = "NanumGothic"
plt.rcParams['font.sans-serif'] = ['NanumGothic', 'sans_serif']
plt.rcParams['axes.unicode_minus'] = False #마이너스(-) 기호가 깨지지 않도록 유니코드 마이너스 비활성화

print("현재 폰트: ",plt.rcParams['font.family']) #현재 적용된 폰트 이름 출력

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
현재 폰트:  ['NanumGothic']


In [88]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import requests
import xml.etree.ElementTree as ET
import warnings #코드 실행 중 발생하는 불필요한 경고 메시지를 숨김
warnings.filterwarnings('ignore')
import pprint

### API에서 직접 데이터 수집

In [89]:
#XML방식
api_key = '4263536b446a697738334a55705553'
start=1 #시작 행 번호
end=1000 #끝 행 번호

url = (f'http://openapi.seoul.go.kr:8088/{api_key}/xml'
      f'/CardSubwayTime/{start}/{end}/202601/경원선/광운대')
#url = (f'http://openapi.seoul.go.kr:8088/{api_key}/xml'
#      f'/TbUseMonthstatusView/{start}/{end}/')

response = requests.get(url, timeout=10) #HTTP GET 요청을 보내고 응답을 받음, 최대 10초 대기
response

#XML 바이트 -> 트리 구조 파싱
root = ET.fromstring(response.content)

#전체 건수 확인 (list_total_count 태그)
total_tag = root.find('.//list_total_count') #.// 하위 해당 태그를 모두 찾음
if total_tag is not None:
  print(f'전체 데이터 건수: {total_tag.text}')

rows = []
for row in root.findall('.//row'): #모든 row태그 탐색
  rows.append({child.tag:child.text for child in row}) #각 <row>태그 내의 자식 태그(컬럼명)와 텍스트(설명)

df=pd.DataFrame(rows)

전체 데이터 건수: 1


In [90]:
df

,USE_MM,SBWY_ROUT_LN_NM,STTN,HR_4_GET_ON_NOPE,HR_4_GET_OFF_NOPE,HR_5_GET_ON_NOPE,HR_5_GET_OFF_NOPE,HR_6_GET_ON_NOPE,HR_6_GET_OFF_NOPE,HR_7_GET_ON_NOPE,...,HR_23_GET_OFF_NOPE,HR_0_GET_ON_NOPE,HR_0_GET_OFF_NOPE,HR_1_GET_ON_NOPE,HR_1_GET_OFF_NOPE,HR_2_GET_ON_NOPE,HR_2_GET_OFF_NOPE,HR_3_GET_ON_NOPE,HR_3_GET_OFF_NOPE,JOB_YMD
0,202601,경원선,광운대,2121,1,6310,1169,14307,3011,31022,...,9130,163,5117,0,2,0,0,0,0,20260203


In [91]:
df.info()
missing_count = df.isnull().sum()
print(missing_count)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1 entries, 0 to 0
Data columns (total 52 columns):
 #   Column              Non-Null Count  Dtype 
---  ------              --------------  ----- 
 0   USE_MM              1 non-null      object
 1   SBWY_ROUT_LN_NM     1 non-null      object
 2   STTN                1 non-null      object
 3   HR_4_GET_ON_NOPE    1 non-null      object
 4   HR_4_GET_OFF_NOPE   1 non-null      object
 5   HR_5_GET_ON_NOPE    1 non-null      object
 6   HR_5_GET_OFF_NOPE   1 non-null      object
 7   HR_6_GET_ON_NOPE    1 non-null      object
 8   HR_6_GET_OFF_NOPE   1 non-null      object
 9   HR_7_GET_ON_NOPE    1 non-null      object
 10  HR_7_GET_OFF_NOPE   1 non-null      object
 11  HR_8_GET_ON_NOPE    1 non-null      object
 12  HR_8_GET_OFF_NOPE   1 non-null      object
 13  HR_9_GET_ON_NOPE    1 non-null      object
 14  HR_9_GET_OFF_NOPE   1 non-null      object
 15  HR_10_GET_ON_NOPE   1 non-null      object
 16  HR_10_GET_OFF_NOPE  1 non-null

In [92]:
#JSON방식
url = (f'http://openapi.seoul.go.kr:8088/{api_key}/json'
      f'/CardSubwayTime/{start}/{end}/202601/경원선/광운대')

data = requests.get(url, timeout=10).json() #json 형식으로 자동 파싱
#pprint.pprint(data)

svc=data.get('CardSubwayTime',{}) #키의 값을 가져오기
print(f'전체 데이터 건수: {svc.get("list_total_count", "알수없음")}')

df_j = pd.DataFrame(svc.get('row',[]))

전체 데이터 건수: 1


In [93]:
df_j

,USE_MM,SBWY_ROUT_LN_NM,STTN,HR_4_GET_ON_NOPE,HR_4_GET_OFF_NOPE,HR_5_GET_ON_NOPE,HR_5_GET_OFF_NOPE,HR_6_GET_ON_NOPE,HR_6_GET_OFF_NOPE,HR_7_GET_ON_NOPE,...,HR_23_GET_OFF_NOPE,HR_0_GET_ON_NOPE,HR_0_GET_OFF_NOPE,HR_1_GET_ON_NOPE,HR_1_GET_OFF_NOPE,HR_2_GET_ON_NOPE,HR_2_GET_OFF_NOPE,HR_3_GET_ON_NOPE,HR_3_GET_OFF_NOPE,JOB_YMD
0,202601,경원선,광운대,2121.0,1.0,6310.0,1169.0,14307.0,3011.0,31022.0,...,9130.0,163.0,5117.0,0.0,2.0,0.0,0.0,0.0,0.0,20260203


In [98]:
#csv 파일로 저장
BASE = '/content/drive/MyDrive/KWU/bigdata/eda/'
PATH_DATA = BASE + 'CardSubwayTime.csv'

df_j.to_csv(PATH_DATA, index=False, encoding='utf-8-sig')